# Figure 2 — Multi-Scale Entropy Paradox in Chordoma Stem Cells

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aistanbulresearch/msep/blob/main/notebooks/figures/figure2_paradox.ipynb)

Reproduces **Figure 2** from Çavuş & Kuşkucu (2026): the multi-scale paradox where chordoma CSC simultaneously exhibit the highest per-cell Shannon entropy among all cell types *and* the lowest across-cells EMT CV across 12 cancer types.

**Panels generated in this notebook**

| Panel | What it shows | msep function |
|---|---|---|
| A | Per-cell Shannon entropy violin by cell type — CSC ranks highest | `msep.plot_entropy_violin` |
| B | Across-cells CV by pathway for CSC_TBXT+ — EMT lowest, immune evasion highest | custom bar plot on `result.pathway_cv` |
| C | Heatmap of CV (cell types × pathways) | `msep.plot_pathway_cv_heatmap` |
| D | Paradox scatter (median entropy vs EMT CV) | `msep.plot_paradox` |

**Data source.** The cell below defaults to the bundled synthetic demo (`msep.datasets.load_example()`), which reproduces the *qualitative* paradox on a 500-cell toy AnnData in under a minute. To regenerate the paper-exact figure from the 106,584-cell Arrieta 2025 cohort, follow the **Real data swap-in** section at the end of the notebook.

**Runtime estimate.** Demo mode: ~1 minute (CPU). Real-data mode: ~15 minutes (Colab Pro, CPU) after the initial Zenodo download.

## 1. Install

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'msep[plotting]>=1.1.0'

## 2. Load the dataset

Default: the 500-cell synthetic demo. See the last section for how to swap in the real Arrieta 2025 cohort.

In [ ]:
import msep

adata = msep.datasets.load_example(n_cells=500, seed=42)
print(adata)
print('\nCell-type composition:')
print(adata.obs['cell_type'].value_counts())

## 3. Run the MSEP pipeline

A single call computes per-cell Shannon entropy, pathway-level CV, bootstrap confidence intervals, and per-gene CV for each of the four built-in cancer-defense pathways.

In [ ]:
result = msep.profile(
    adata,
    pathways='cancer_defense',
    cell_type_key='cell_type',
    compute_bootstrap=True,
    n_boot=500,
    compute_gene_cv=True,
)
print(result)
print('\nParadox summary:')
result.paradox_summary.round(3)

## 4. Panel A — Per-cell Shannon entropy violin

Expected pattern: CSC_TBXT+ sits at the top of the ranking. In the full Arrieta 2025 cohort this corresponds to the paper's reported median of 9.97 bits.

In [ ]:
import matplotlib.pyplot as plt

fig_a = msep.plot_entropy_violin(result, figsize=(10, 4.5))
fig_a.savefig('figure2_panelA_entropy_violin.pdf', bbox_inches='tight')
fig_a.savefig('figure2_panelA_entropy_violin.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Panel B — Pathway CV bar chart for CSC

Among the four cancer-defense pathways, CSC show the lowest CV for EMT (tight coordination) and the highest CV for immune evasion (heterogeneous expression). Housekeeping serves as a reference.

In [ ]:
import pandas as pd

csc_cv = (
    result.pathway_cv
    .query('cell_type == "CSC_TBXT+"')
    .set_index('pathway')['cv']
    .reindex(['emt', 'ferroptosis', 'immune_evasion', 'housekeeping'])
)

fig_b, ax_b = plt.subplots(figsize=(6, 4))
colors = ['#2ECC71', '#E74C3C', '#3498DB', '#95A5A6']
bars = ax_b.bar(csc_cv.index, csc_cv.values, color=colors, edgecolor='black', linewidth=0.8)
ax_b.set_ylabel('Across-cells CV (lower = more coordinated)')
ax_b.set_title('CSC_TBXT+ pathway coordination ranking')
for bar, val in zip(bars, csc_cv.values):
    ax_b.text(bar.get_x() + bar.get_width()/2, val + 0.05, f'{val:.2f}',
              ha='center', fontsize=10, fontweight='bold')
ax_b.spines[['top', 'right']].set_visible(False)
fig_b.tight_layout()
fig_b.savefig('figure2_panelB_csc_pathway_cv.pdf', bbox_inches='tight')
fig_b.savefig('figure2_panelB_csc_pathway_cv.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Panel C — Cell type × pathway CV heatmap

A bird's-eye view of population coordination across the whole cellular landscape. Low CV (lighter cells) marks tight coordination.

In [ ]:
fig_c = msep.plot_pathway_cv_heatmap(result, figsize=(7, 4))
fig_c.savefig('figure2_panelC_cv_heatmap.pdf', bbox_inches='tight')
fig_c.savefig('figure2_panelC_cv_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Panel D — Paradox scatter (entropy vs CV)

Each dot is a cell type. The **top-left quadrant** is the `individually diverse, collectively disciplined` phenotype — high per-cell entropy + low across-cells CV. CSC_TBXT+ should land there.

In [ ]:
fig_d = msep.plot_paradox(result, cv_pathway='emt', figsize=(7, 5))
fig_d.savefig('figure2_panelD_paradox.pdf', bbox_inches='tight')
fig_d.savefig('figure2_panelD_paradox.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Combined 2×2 figure

Replicates the 4-panel layout of Figure 2 from the paper.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A — entropy violin (embedded)
import seaborn as sns
df_e = result.per_cell_entropy[['cell_type', 'entropy_global']].dropna()
order = df_e.groupby('cell_type')['entropy_global'].median().sort_values(ascending=False).index.tolist()
sns.violinplot(data=df_e, x='cell_type', y='entropy_global', order=order,
               inner='quartile', cut=0, palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('A · Per-cell Shannon entropy')
axes[0, 0].set_ylabel('Entropy (bits)')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=45)

# Panel B — CSC pathway CV bar
axes[0, 1].bar(csc_cv.index, csc_cv.values, color=colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(csc_cv.values):
    axes[0, 1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
axes[0, 1].set_title('B · CSC_TBXT+ pathway CV')
axes[0, 1].set_ylabel('CV')
axes[0, 1].spines[['top', 'right']].set_visible(False)

# Panel C — heatmap
pivot = result.pathway_cv.pivot(index='cell_type', columns='pathway', values='cv')
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd_r', linewidths=0.5,
            vmin=0, cbar_kws={'label': 'CV'}, ax=axes[1, 0])
axes[1, 0].set_title('C · CV heatmap: cell type × pathway')

# Panel D — paradox scatter
summary = result.paradox_summary
axes[1, 1].scatter(summary['median_entropy'], summary['cv_emt'],
                   s=120, edgecolors='black', linewidths=0.8,
                   c=range(len(summary)), cmap='tab10', zorder=3)
for ct in summary.index:
    axes[1, 1].annotate(ct, (summary.loc[ct, 'median_entropy'], summary.loc[ct, 'cv_emt']),
                        textcoords='offset points', xytext=(6, 6), fontsize=9)
axes[1, 1].set_xlabel('Median per-cell entropy (bits)')
axes[1, 1].set_ylabel('EMT across-cells CV')
axes[1, 1].set_title('D · Multi-scale paradox')

fig.suptitle('Figure 2 — Multi-scale entropy paradox in chordoma CSC', fontsize=14, y=1.00)
fig.tight_layout()
fig.savefig('figure2_combined.pdf', bbox_inches='tight')
fig.savefig('figure2_combined.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nSaved:')
!ls -la figure2_*.pdf figure2_*.png 2>/dev/null | awk '{print "  ", $NF}'

## 9. Quantitative comparison with the paper

On the demo dataset the *rankings* will match the paper, but absolute magnitudes are smaller (200 CSC vs 6,730 CSC, controlled dispersion).

In [ ]:
paper_values = pd.DataFrame({
    'metric': [
        'CSC median per-cell entropy (bits)',
        'CSC EMT CV (lowest among pathways)',
        'CSC ferroptosis CV',
        'CSC immune evasion CV (highest)',
        'CSC housekeeping CV',
        'TBXT CV in CSC (gene-level)',
    ],
    'paper_full_cohort': [9.97, 4.632, 5.387, 8.548, 1.743, 1.02],
})

demo_entropy = result.per_cell_entropy.query('cell_type == "CSC_TBXT+"')['entropy_global'].median()
tbxt_col = adata.var_names.get_loc('TBXT')
csc_mask = adata.obs['cell_type'] == 'CSC_TBXT+'
tbxt_vals = adata.X.toarray()[csc_mask, tbxt_col]
tbxt_cv = tbxt_vals.std() / tbxt_vals.mean() if tbxt_vals.mean() > 0 else float('nan')

paper_values['this_notebook_demo'] = [
    round(demo_entropy, 3),
    round(csc_cv.get('emt', float('nan')), 3),
    round(csc_cv.get('ferroptosis', float('nan')), 3),
    round(csc_cv.get('immune_evasion', float('nan')), 3),
    round(csc_cv.get('housekeeping', float('nan')), 3),
    round(tbxt_cv, 3),
]
paper_values

## 10. Real-data swap-in (optional)

To regenerate Figure 2 with paper-exact numbers, replace the `load_example()` call in Section 2 with a loader that fetches the Arrieta et al. 2025 cohort from Zenodo. The rest of the notebook is dataset-agnostic.

```python
# --- Uncomment and fill in the Zenodo record ID ---
# import scanpy as sc, anndata as ad, urllib.request, gzip, shutil, os, tarfile
# ZENODO_URL = 'https://zenodo.org/records/<RECORD_ID>/files/<ARCHIVE_NAME>.tar.gz'
# urllib.request.urlretrieve(ZENODO_URL, 'arrieta2025.tar.gz')
# with tarfile.open('arrieta2025.tar.gz') as t: t.extractall('arrieta2025/')
#
# # Read each patient's filtered_feature_bc_matrix.h5 and concatenate
# samples = ['NU02757','NU02854','NU03174','NU03231','NU03287','NU03372']
# adatas = []
# for s in samples:
#     a = sc.read_10x_h5(f'arrieta2025/{s}/filtered_feature_bc_matrix.h5')
#     a.obs['patient_id'] = s
#     a.var_names_make_unique()
#     adatas.append(a)
# adata = ad.concat(adatas, join='outer', label='patient_id', keys=samples, merge='unique')
#
# # QC: 200+ genes/cell, 3+ cells/gene, mt<20%, Scrublet doublets, 98th pct genes/cell cap
# sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
# adata = adata[adata.obs['n_genes_by_counts'] >= 200]
# adata.var['mt'] = adata.var_names.str.startswith('MT-')
# adata = adata[adata.obs['pct_counts_mt'] < 20]
# adata.layers['raw_counts'] = adata.X.copy()
#
# # Normalise, HVGs, PCA, Harmony, Leiden, annotate cell types per §2.3 of the paper.
# # See examples/chordoma_msep.ipynb for the full annotation pipeline.
```

The full pipeline (QC → Harmony → Leiden → cell-type annotation via canonical markers) is implemented in [`examples/chordoma_msep.ipynb`](../../examples/chordoma_msep.ipynb). After substituting the real `adata`, the cells in Sections 3–8 of this notebook run unchanged and produce the paper-exact figure.

**Data availability:**
- Arrieta et al. 2025 single-cell RNA-seq: deposited in Zenodo (DOI available via the *Neuro-Oncology* article)
- Halvorsen et al. 2023 bulk RNA-seq (used in Figure 6): GEO `GSE205457`
- Pan-cancer contexts (Figure 3): CellxGene Census v2025-11-08